## Extracción de datos de empresas constituidas desde la API del INE

### Objetivo
Este notebook extrae y estructura los datos de **sociedades mercantiles constituidas** en España a través de la API pública TEMPUS del INE (tabla 13913). 

Los datos se desglosan por **territorio**, **tipo societario** (S.A., S.L., S. Comanditarias y Colectivas, y su agregado Mercantiles) y **periodo mensual**, obteniendo tanto el número de sociedades como el capital suscrito (en euros).

### Metodología
1. **Conexión a la API del INE** — Llamada al endpoint `/DATOS_TABLA/13913`.
2. **Extracción y desanidado** — Recorrido de la respuesta JSON para poblar un diccionario con las columnas: `id_const`, `territorio`, `id_tiempo`, `tipo`, `numero_sociedades` y `capital`.
3. **Limpieza en línea** — Corrección de escalas (FK_Escala 4 → multiplicar por 1000) y concatenación de filas de número de sociedades y capital en un mismo registro.
4. **Exportación** — Volcado a CSV en `../files/data_raw/empresas_constituidas.csv`.

### Contexto del proyecto
Estos datos se integran en un análisis de **resiliencia empresarial en España**, donde se cruzarán con disoluciones de empresas e IPC para estudiar el impacto del ciclo económico en la creación de empresas.

In [ ]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación del módulo de conexión a la API
from src.api import conection_api as conection_api
from src.api.config import API_URLS

In [ ]:
url_const = API_URLS["constituidas"]  #Constituidas

In [ ]:
data_const = conection_api.llamada_api(url_const)

In [ ]:
data_const

In [ ]:
for dato in data_const:
    for serie in dato['Data']:
        print(f'{dato['Nombre']}, {serie['FK_TipoDato']}, {serie['FK_Periodo']}, {serie['Anyo']}, {serie['Valor']}')

In [ ]:
empresas_constituidas = { 
    'id_const': [], 
    'territorio': [], 
    'id_tiempo': [], 
    'tipo': [], 
    'numero_sociedades': [], 
    'capital': []            
}
contador = 1

for serie in data_const:
    nombre_completo = serie['Nombre']
    
    if "nacional" not in nombre_completo.lower():
        partes = nombre_completo.split('.', 3)
        territorio = partes[0].strip()
        tipo = partes[2].strip()
        unidad = partes[3].replace('.', '').strip() 

        # CORRECCIÓN EN LÍNEA: Si el tipo se quedó cortado en "S", le ponemos el nombre completo
        if tipo == "S":
            tipo = "S. Comanditarias y S. Colectivas"

        for data in serie['Data']:
            valor = data['Valor'] * 1000 if serie.get("FK_Escala") == 4 else data['Valor']
            id_tiempo = str(data['Anyo']) + str(data['FK_Periodo']).zfill(2)

            # COMPROBACIÓN: ¿Ya hemos añadido esta misma combinación?
            existe = False
            for i in range(len(empresas_constituidas['territorio'])):
                if (empresas_constituidas['territorio'][i] == territorio and 
                    empresas_constituidas['id_tiempo'][i] == id_tiempo and 
                    empresas_constituidas['tipo'][i] == tipo):
                    
                    if "capital" in unidad.lower():
                        empresas_constituidas['capital'][i] = int(valor)
                    else:
                        empresas_constituidas['numero_sociedades'][i] = int(valor)
                    existe = True
                    break
            
            # Si NO existe, creamos una nueva fila
            if not existe:
                empresas_constituidas['id_const'].append(contador)
                empresas_constituidas['territorio'].append(territorio)
                empresas_constituidas['id_tiempo'].append(id_tiempo)
                empresas_constituidas['tipo'].append(tipo)
                
                if "capital" in unidad.lower():
                    empresas_constituidas['capital'].append(int(valor))
                    empresas_constituidas['numero_sociedades'].append(None)
                else:
                    empresas_constituidas['capital'].append(None)
                    empresas_constituidas['numero_sociedades'].append(int(valor))
                
                contador += 1

In [ ]:
empresas_constituidas

In [ ]:
pd.DataFrame(empresas_constituidas).to_csv('../files/data_raw/empresas_constituidas.csv', index=False)